## Solving a predator-prey model using Jax

**Last updated:** 2025-05-01

This notebook demonstrates running the predator-prey model using the Jax model.

In [ ]:
from pathlib import Path

import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pytensor
import pytensor.tensor as pt
from pytensor.graph import Apply, Op
from pytensor.link.jax.dispatch import jax_funcify
from rich import print

from pymcsimmod import model
from pymcsimmod.parser import ModelParser

First, we load the model and parse it.

In [ ]:
filename = Path("../../data/pred_prey.model")
assert filename.exists()
print(filename.absolute().resolve())

In [ ]:
parser = ModelParser()
parsed_model = parser.parse(filename.read_text())

Convert the parsed model into Jax code. This will be refactored in the future so that library can do this itself, instead of in the notebook:

In [ ]:
# Extract parameters from parsed_model
parameters = {}
for section in parsed_model.sections:
    if isinstance(section, model.Statement) and isinstance(section.rhs, model.Number):
        parameters[section.lhs.name] = section.rhs.value

# Extract initial conditions
initial_conditions = {}
for section in parsed_model.sections:
    if isinstance(section, model.InitializeSection):
        for statement in section.statements:
            initial_conditions[statement.lhs.name] = statement.rhs.value

# Extract dynamics
dynamics = {}
for section in parsed_model.sections:
    if isinstance(section, model.DynamicsSection):
        for statement in section.statements:
            variable = statement.lhs.identifier.name  # e.g., 'x' or 'y'
            dynamics[variable] = statement.rhs  # Store the MathematicalExpression

# Helper function to evaluate MathematicalExpression
def evaluate_expression(expr, variables):
    if isinstance(expr, model.Identifier):
        return variables[expr.name]
    elif isinstance(expr, model.Number):
        return expr.value
    elif isinstance(expr, model.MathematicalExpression):
        lhs = evaluate_expression(expr.lhs, variables)
        rhs = evaluate_expression(expr.rhs, variables)
        if expr.operator == '+':
            return lhs + rhs
        elif expr.operator == '-':
            return lhs - rhs
        elif expr.operator == '*':
            return lhs * rhs
        elif expr.operator == '/':
            return lhs / rhs
    raise ValueError("Unsupported expression type")

# Define generic ODE function
def generic_ode(t, y, args):
    # Unpack the arguments
    init_conds = {state: y[i] for i, state in enumerate(dynamics.keys())}
    variables = {**args}
    variables.update(init_conds)

    # Combine dynamics into list
    dydt = [evaluate_expression(dynamics[state], variables) for state in dynamics.keys()]
    
    return jnp.array(dydt)

# Run the predator-prey model using the parsed model
# Unpack parameters and initial conditions
x0, y0 = initial_conditions['x'], initial_conditions['y']

# Time span for the simulation
t0 = 0.0  # Start time
t1 = 50.0  # End time

# Create the ODE term
ode_term = diffrax.ODETerm(generic_ode)

# Initial state
y_init = jnp.array([x0, y0])

# Solver configuration
solver = diffrax.Dopri5()
saveat = diffrax.SaveAt(ts=jnp.linspace(t0, t1, 500))

Solve the ODE:

In [ ]:
solution = diffrax.diffeqsolve(
    ode_term,
    solver,
    t0=t0,
    t1=t1,
    dt0=0.1,
    y0=y_init,
    args=parameters,  # Pass parameters as args
    saveat=saveat,
)

Plot the results:

In [ ]:
plt.plot(solution.ts, solution.ys[:, 0], label="Rabbits (x)")
plt.plot(solution.ts, solution.ys[:, 1], label="Foxes (y)")
plt.xlabel("Time (days)")
plt.ylabel("Population (1000s)")
plt.legend()
plt.title("Predator-Prey Model (Lotka-Volterra)")
plt.show()